# Lesson 04 - Pola Desain Penggunaan Alat

Dalam pelajaran ini Anda akan mempelajari pola desain **Penggunaan Alat** untuk agen AI menggunakan Microsoft Agent Framework (Python). Kami membahas:

- Mendefinisikan alat fungsi dengan dekorator `@tool` dan parameter bertipe
- Menyediakan skema alat agar model tahu apa yang dilakukan setiap alat
- Mengendalikan eksekusi alat dengan `approval_mode`
- Mengembalikan **output terstruktur** melalui model Pydantic dan `response_format`

Skenarionya adalah agen **pemesan perjalanan** yang dapat mencari tujuan, memeriksa ketersediaan, dan mengambil informasi penerbangan.


## Pengaturan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mendefinisikan Alat dengan Dekorator @tool

Dekorator `@tool` mengubah fungsi Python biasa menjadi alat yang dapat dipanggil oleh agen.  
Poin-poin utama:

- **Docstring** menjadi deskripsi alat yang dilihat model.  
- **Anotasi tipe** (termasuk `Annotated` dengan deskripsi) mendefinisikan skema alat.  
- `approval_mode` mengontrol apakah pengguna harus menyetujui setiap panggilan sebelum dijalankan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Membuat Agen dengan Beberapa Alat

Berikan ketiga alat kepada klien sehingga model dapat menggunakan alat mana pun yang dibutuhkan untuk menjawab pertanyaan pengguna.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Keluaran Terstruktur dengan Alat

Dengan mengatur `response_format` ke model Pydantic, agen dipaksa untuk mengembalikan objek JSON yang bertipe dengan baik alih-alih teks bebas. Ini berguna ketika kode selanjutnya perlu mengonsumsi hasil secara programatis.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Pola Persetujuan Alat

Parameter `approval_mode` pada `@tool` mengontrol apakah panggilan alat memerlukan persetujuan manusia sebelum dijalankan:

| Mode | Perilaku |
|---|---|
| `"never_require"` | Alat berjalan otomatis — tidak perlu konfirmasi pengguna. |
| `"always_require"` | Setiap panggilan harus disetujui oleh pengguna sebelum dijalankan. |

Gunakan `"always_require"` untuk alat yang memiliki efek samping (misalnya memesan penerbangan, mengenakan biaya pada kartu kredit) agar manusia tetap terlibat dalam prosesnya.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Ringkasan

Dalam pelajaran ini Anda belajar bagaimana:

1. **Mendefinisikan alat** menggunakan dekorator `@tool` dengan parameter bertipe dan docstring yang berfungsi sebagai skema alat.
2. **Menyusun beberapa alat** sehingga agen dapat memanggilnya secara berurutan untuk menjawab kueri kompleks.
3. **Mengembalikan output terstruktur** dengan melewatkan model Pydantic sebagai `response_format`.
4. **Mengontrol persetujuan alat** dengan `approval_mode` untuk menjaga keterlibatan manusia dalam operasi yang sensitif.

Polanya ini membentuk dasar untuk membangun agen yang andal dan siap produksi yang dapat berinteraksi dengan sistem eksternal dengan aman.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berusaha untuk akurasi, harap diingat bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau salah tafsir yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
